CineMatch - Database Integration
In this notebook we design and implement the relational database for CineMatch. 
The database stores all movie metadata, user information, ratings, genres and interaction logs.
It is built using SQLite and integrated with the existing AI recommendation system.

In [1]:
import sqlite3
import pandas as pd
import numpy as np
import os

# create database file in current folder
conn = sqlite3.connect('cinematch.db')
cursor = conn.cursor()

print("Database connected successfully!")
print(f"Database file created at: {os.path.abspath('cinematch.db')}")

Database connected successfully!
Database file created at: C:\Users\mzcs2\Downloads\Cinematch(1)\cinematch.db


Step 2-Creating tables


In [2]:
# Create all tables
cursor.executescript('''
    CREATE TABLE IF NOT EXISTS users (
        user_id INTEGER PRIMARY KEY,
        age INTEGER,
        gender TEXT,
        occupation TEXT,
        zip_code TEXT
    );

    CREATE TABLE IF NOT EXISTS movies (
        movie_id INTEGER PRIMARY KEY,
        title TEXT NOT NULL,
        release_year INTEGER,
        imdb_url TEXT
    );

    CREATE TABLE IF NOT EXISTS genres (
        genre_id INTEGER PRIMARY KEY AUTOINCREMENT,
        genre_name TEXT UNIQUE NOT NULL
    );

    CREATE TABLE IF NOT EXISTS movie_genres (
        movie_id INTEGER,
        genre_id INTEGER,
        PRIMARY KEY (movie_id, genre_id),
        FOREIGN KEY (movie_id) REFERENCES movies(movie_id),
        FOREIGN KEY (genre_id) REFERENCES genres(genre_id)
    );

    CREATE TABLE IF NOT EXISTS ratings (
        rating_id INTEGER PRIMARY KEY AUTOINCREMENT,
        user_id INTEGER,
        movie_id INTEGER,
        score INTEGER,
        rated_at TIMESTAMP,
        FOREIGN KEY (user_id) REFERENCES users(user_id),
        FOREIGN KEY (movie_id) REFERENCES movies(movie_id)
    );

    CREATE TABLE IF NOT EXISTS interaction_logs (
        log_id INTEGER PRIMARY KEY AUTOINCREMENT,
        user_id INTEGER,
        query_movie TEXT,
        recommended_movies TEXT,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        FOREIGN KEY (user_id) REFERENCES users(user_id)
    );
''')

conn.commit()
print("All tables created successfully!")

All tables created successfully!


Step 3- Loading data into these tables

In [3]:
# Load the raw data files
ratings_df = pd.read_csv('ml-100k/ml-100k/u.data',
                          sep='\t',
                          names=['user_id', 'movie_id', 'rating', 'timestamp'])

users_df = pd.read_csv('ml-100k/ml-100k/u.user',
                        sep='|',
                        names=['user_id', 'age', 'gender', 'occupation', 'zip_code'])

genre_columns = ['unknown', 'Action', 'Adventure', 'Animation', 'Childrens',
                 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy',
                 'FilmNoir', 'Horror', 'Musical', 'Mystery', 'Romance',
                 'SciFi', 'Thriller', 'War', 'Western']

movies_df = pd.read_csv('ml-100k/ml-100k/u.item',
                         sep='|',
                         encoding='latin-1',
                         names=['movie_id', 'title', 'release_date', 'video_date',
                                'imdb_url'] + genre_columns)

print(f"Users loaded: {len(users_df)}")
print(f"Movies loaded: {len(movies_df)}")
print(f"Ratings loaded: {len(ratings_df)}")

Users loaded: 943
Movies loaded: 1682
Ratings loaded: 100000


In [4]:
# Insert users
users_df.to_sql('users', conn, if_exists='replace', index=False)
print(f"Inserted {len(users_df)} users")

# Insert movies
movies_df[['movie_id', 'title', 'release_date', 'imdb_url']].rename(
    columns={'release_date': 'release_year'}
).to_sql('movies', conn, if_exists='replace', index=False)
print(f"Inserted {len(movies_df)} movies")

# Insert genres
genre_list = [(i+1, genre) for i, genre in enumerate(genre_columns)]
cursor.executemany("INSERT OR IGNORE INTO genres (genre_id, genre_name) VALUES (?, ?)", genre_list)
conn.commit()
print(f"Inserted {len(genre_list)} genres")

# Insert movie_genres (many-to-many)
movie_genre_rows = []
for _, row in movies_df.iterrows():
    for i, genre in enumerate(genre_columns):
        if row[genre] == 1:
            movie_genre_rows.append((int(row['movie_id']), i+1))

cursor.executemany("INSERT OR IGNORE INTO movie_genres (movie_id, genre_id) VALUES (?, ?)", movie_genre_rows)
conn.commit()
print(f"Inserted {len(movie_genre_rows)} movie-genre relationships")

# Insert ratings
ratings_df.rename(columns={'rating': 'score', 'timestamp': 'rated_at'})[
    ['user_id', 'movie_id', 'score', 'rated_at']
].to_sql('ratings', conn, if_exists='replace', index=False)
print(f"Inserted {len(ratings_df)} ratings")

Inserted 943 users
Inserted 1682 movies
Inserted 19 genres
Inserted 2893 movie-genre relationships
Inserted 100000 ratings


In [5]:
# Verify all tables have data
tables = ['users', 'movies', 'genres', 'movie_genres', 'ratings', 'interaction_logs']

print("Table row counts:")
print("-" * 30)
for table in tables:
    cursor.execute(f"SELECT COUNT(*) FROM {table}")
    count = cursor.fetchone()[0]
    print(f"{table:20} {count:>8} rows")

Table row counts:
------------------------------
users                     943 rows
movies                   1682 rows
genres                     19 rows
movie_genres             2893 rows
ratings                100000 rows
interaction_logs            0 rows


Step 4-SQL queries
We now run meaningful SQL queries on the database to extract useful insights


In [7]:
# Query 1: Top 10 highest rated movies (minimum 50 ratings)
query1 = '''
    SELECT m.title, 
           ROUND(AVG(r.score), 2) AS avg_rating,
           COUNT(*) AS total_ratings
    FROM movies m
    JOIN ratings r ON m.movie_id = r.movie_id
    GROUP BY m.movie_id, m.title
    HAVING COUNT(*) >= 50
    ORDER BY avg_rating DESC
    LIMIT 10
'''

result1 = pd.read_sql_query(query1, conn)
print("Top 10 Highest Rated Movies (min 50 ratings):\n")
print(result1.to_string(index=False))

Top 10 Highest Rated Movies (min 50 ratings):

                                                 title  avg_rating  total_ratings
                                 Close Shave, A (1995)        4.49            112
                            Wrong Trousers, The (1993)        4.47            118
                               Schindler's List (1993)        4.47            298
                                     Casablanca (1942)        4.46            243
                      Shawshank Redemption, The (1994)        4.45            283
Wallace & Gromit: The Best of Aardman Animation (1996)        4.45             67
                            Usual Suspects, The (1995)        4.39            267
                                    Rear Window (1954)        4.39            209
                                      Star Wars (1977)        4.36            583
                                   12 Angry Men (1957)        4.34            125


In [8]:
# Query 2: Rating history for a specific user
query2 = '''
    SELECT m.title,
           r.score,
           r.rated_at
    FROM ratings r
    JOIN movies m ON r.movie_id = m.movie_id
    WHERE r.user_id = 1
    ORDER BY r.score DESC
    LIMIT 10
'''

result2 = pd.read_sql_query(query2, conn)
print("Top 10 movies rated by User 1:\n")
print(result2.to_string(index=False))

Top 10 movies rated by User 1:

                                                     title  score  rated_at
                                      Groundhog Day (1993)      5 875072442
                                       Delicatessen (1991)      5 889751711
                                   Pillow Book, The (1995)      5 874965970
Horseman on the Roof, The (Hussard sur le toit, Le) (1995)      5 878542738
                          Shawshank Redemption, The (1994)      5 875072404
                       Star Trek: The Wrath of Khan (1982)      5 878543541
    Wallace & Gromit: The Best of Aardman Animation (1996)      5 875072173
                                 Breaking the Waves (1996)      5 887431921
                                 Three Colors: Blue (1993)      5 875072370
                    Good, The Bad and The Ugly, The (1966)      5 876892701


In [9]:
# Query 3: Find all movies belonging to a specific genre
query3 = '''
    SELECT m.title, g.genre_name
    FROM movies m
    JOIN movie_genres mg ON m.movie_id = mg.movie_id
    JOIN genres g ON mg.genre_id = g.genre_id
    WHERE g.genre_name = 'Action'
    ORDER BY m.title
    LIMIT 10
'''

result3 = pd.read_sql_query(query3, conn)
print("Action movies in the database:\n")
print(result3.to_string(index=False))

Action movies in the database:

                                      title genre_name
3 Ninjas: High Noon At Mega Mountain (1998)     Action
                          Abyss, The (1989)     Action
       Adventures of Robin Hood, The (1938)     Action
                  African Queen, The (1951)     Action
                       Air Force One (1997)     Action
                               Alien (1979)     Action
                             Alien 3 (1992)     Action
                 Alien: Resurrection (1997)     Action
                              Aliens (1986)     Action
                     American Strays (1996)     Action


In [10]:
# Query 4: Top 10 most active users
query4 = '''
    SELECT r.user_id,
           u.occupation,
           COUNT(*) AS total_ratings,
           ROUND(AVG(r.score), 2) AS avg_rating
    FROM ratings r
    JOIN users u ON r.user_id = u.user_id
    GROUP BY r.user_id
    ORDER BY total_ratings DESC
    LIMIT 10
'''

result4 = pd.read_sql_query(query4, conn)
print("Top 10 Most Active Users:\n")
print(result4.to_string(index=False))

Top 10 Most Active Users:

 user_id occupation  total_ratings  avg_rating
     405 healthcare            737        1.83
     655 healthcare            685        2.91
      13   educator            636        3.10
     450   educator            540        3.86
     276    student            518        3.47
     416    student            493        3.85
     537   engineer            490        2.87
     303    student            484        3.37
     234    retired            480        3.12
     393    student            448        3.34


In [11]:
# Query 5: Genre popularity by number of movies
query5 = '''
    SELECT g.genre_name,
           COUNT(*) AS movie_count
    FROM genres g
    JOIN movie_genres mg ON g.genre_id = mg.genre_id
    GROUP BY g.genre_name
    ORDER BY movie_count DESC
'''

result5 = pd.read_sql_query(query5, conn)
print("Genre Popularity:\n")
print(result5.to_string(index=False))

Genre Popularity:

 genre_name  movie_count
      Drama          725
     Comedy          505
   Thriller          251
     Action          251
    Romance          247
  Adventure          135
  Childrens          122
      Crime          109
      SciFi          101
     Horror           92
        War           71
    Mystery           61
    Musical           56
Documentary           50
  Animation           42
    Western           27
   FilmNoir           24
    Fantasy           22
    unknown            2


In [12]:
# Query 6: Average rating per genre
query6 = '''
    SELECT g.genre_name,
           ROUND(AVG(r.score), 2) AS avg_rating,
           COUNT(*) AS total_ratings
    FROM genres g
    JOIN movie_genres mg ON g.genre_id = mg.genre_id
    JOIN ratings r ON mg.movie_id = r.movie_id
    GROUP BY g.genre_name
    ORDER BY avg_rating DESC
'''

result6 = pd.read_sql_query(query6, conn)
print("Average Rating by Genre:\n")
print(result6.to_string(index=False))

Average Rating by Genre:

 genre_name  avg_rating  total_ratings
   FilmNoir        3.92           1733
        War        3.82           9398
      Drama        3.69          39895
Documentary        3.67            758
    Mystery        3.64           5245
      Crime        3.63           8055
    Romance        3.62          19461
    Western        3.61           1854
  Animation        3.58           3605
      SciFi        3.56          12730
    Musical        3.52           4954
   Thriller        3.51          21872
  Adventure        3.50          13753
     Action        3.48          25589
     Comedy        3.39          29832
  Childrens        3.35           7182
     Horror        3.29           5317
    Fantasy        3.22           1352
    unknown        3.20             10


In [13]:
# Query 7: Cold start - recommend popular movies for new users with no ratings
query7 = '''
    SELECT m.title,
           ROUND(AVG(r.score), 2) AS avg_rating,
           COUNT(*) AS total_ratings
    FROM movies m
    JOIN ratings r ON m.movie_id = r.movie_id
    GROUP BY m.movie_id, m.title
    HAVING COUNT(*) >= 100
    ORDER BY avg_rating DESC, total_ratings DESC
    LIMIT 10
'''

result7 = pd.read_sql_query(query7, conn)
print("Top movies to recommend for new users (cold start):\n")
print(result7.to_string(index=False))

Top movies to recommend for new users (cold start):

                           title  avg_rating  total_ratings
           Close Shave, A (1995)        4.49            112
         Schindler's List (1993)        4.47            298
      Wrong Trousers, The (1993)        4.47            118
               Casablanca (1942)        4.46            243
Shawshank Redemption, The (1994)        4.45            283
      Usual Suspects, The (1995)        4.39            267
              Rear Window (1954)        4.39            209
                Star Wars (1977)        4.36            583
             12 Angry Men (1957)        4.34            125
Silence of the Lambs, The (1991)        4.29            390


In [14]:
# Query 8: Movies liked by users similar to User 1 (using subquery)
query8 = '''
    SELECT m.title,
           COUNT(*) AS liked_by_similar_users,
           ROUND(AVG(r.score), 2) AS avg_score
    FROM ratings r
    JOIN movies m ON r.movie_id = m.movie_id
    WHERE r.score >= 4
    AND r.user_id IN (
        SELECT DISTINCT r2.user_id
        FROM ratings r2
        WHERE r2.movie_id IN (
            SELECT movie_id FROM ratings WHERE user_id = 1 AND score >= 4
        )
        AND r2.user_id != 1
        AND r2.score >= 4
        LIMIT 50
    )
    AND r.movie_id NOT IN (
        SELECT movie_id FROM ratings WHERE user_id = 1
    )
    GROUP BY m.movie_id, m.title
    ORDER BY liked_by_similar_users DESC, avg_score DESC
    LIMIT 10
'''

result8 = pd.read_sql_query(query8, conn)
print("Movies liked by users similar to User 1:\n")
print(result8.to_string(index=False))

Movies liked by users similar to User 1:

                                                                      title  liked_by_similar_users  avg_score
                                          E.T. the Extra-Terrestrial (1982)                      25       4.56
                                     One Flew Over the Cuckoo's Nest (1975)                      25       4.56
                                                    Schindler's List (1993)                      22       4.77
                                                          Casablanca (1942)                      22       4.68
                                                         Rear Window (1954)                      21       4.52
                                               To Kill a Mockingbird (1962)                      20       4.55
                                                              Scream (1996)                      20       4.55
                                                              Gandhi (

Step 5-Logging interactions

In [17]:
# Verify it was logged
result = pd.read_sql_query("SELECT * FROM interaction_logs", conn)
print("\nInteraction Logs Table:\n")
for _, row in result.iterrows():
    print(f"Log ID      : {row['log_id']}")
    print(f"User ID     : {row['user_id']}")
    print(f"Query Movie : {row['query_movie']}")
    print(f"Created At  : {row['created_at']}")
    print(f"Recommended :")
    for movie in row['recommended_movies'].split(', '):
        print(f"              {movie}")
    print("-" * 60)


Interaction Logs Table:

Log ID      : 1
User ID     : 1
Query Movie : Toy Story (1995)
Created At  : 2026-05-08 11:23:51.372621
Recommended :
              Schindler's List (1993)
              Casablanca (1942)
              One Flew Over the Cuckoo's Nest (1975)
              E.T. the Extra-Terrestrial (1982)
              Rear Window (1954)
------------------------------------------------------------


Step 6-Closing the connection

In [18]:
# Always close the connection when done
conn.close()
print("Database connection closed successfully!")
print("Database saved at: cinematch.db")

Database connection closed successfully!
Database saved at: cinematch.db
